# Course 7 · Data Analysis with Python
The full loop: **import → wrangle → explore → model → evaluate**. Every line has a `#` comment.

## Step 1 · Import and inspect

In [ ]:
# os lets Python talk to the operating system (files, folders).
import os
# If the practice data file is not in this folder yet...
if not os.path.exists("applications.csv"):
    # ...run make_data.py to create applications.csv and centers.csv.
    %run make_data.py
# pandas = tables (nickname pd); numpy = fast math (nickname np). These nicknames are standard.
import pandas as pd, numpy as np

# parse_dates turns the "submitted" text column into real dates while loading.
df = pd.read_csv("applications.csv", parse_dates=["submitted"])
centers = pd.read_csv("centers.csv")
# .info() lists each column, its type and how many non-missing values it has.
df.info()

In [ ]:
# .describe() gives count, mean, std, min, quartiles and max for numeric columns.
display(df.describe().round(2))
missing = df.isna().sum()                     # missing values per column
print("Missing:\n", missing[missing > 0])     # show only columns that have some
print("Duplicates:", df.duplicated().sum())  # number of exact duplicate rows

## Step 2 · Wrangling: duplicates, labels, missing values, types

In [ ]:
df = df.drop_duplicates()                                  # remove duplicate rows
df["channel"] = df["channel"].str.strip().str.lower()     # "Walk-In " -> "walk-in"
# Fill missing ages with the median (robust to outliers), then store as whole numbers.
df["age"] = df["age"].fillna(df["age"].median()).astype(int)
# Add center columns (region, lat, lon, staff). how="left" keeps every application.
df = df.merge(centers, on="center_id", how="left")
print(df.shape, df.channel.unique())                      # check the result

## Step 3 · Feature engineering: dates, binning, normalisation, dummies

In [ ]:
df["month"] = df["submitted"].dt.month          # .dt gives date parts: month number 1-12

# BINNING: pd.cut puts numbers into ranges. bins are the edges: (15,24], (24,59], (59,100].
df["age_group"] = pd.cut(df["age"], bins=[15, 24, 59, 100], labels=["16-24", "25-59", "60+"])

# MIN-MAX NORMALISATION: rescale to 0..1 -> (value - min) / (max - min)
bq = df["biometric_quality"]
df["bio_minmax"] = (bq - bq.min()) / (bq.max() - bq.min())

# Z-SCORE STANDARDISATION: how many standard deviations from the mean -> (value - mean) / std
df["staff_z"] = (df["staff"] - df["staff"].mean()) / df["staff"].std()

# ONE-HOT ENCODING (dummies): one 0/1 column per category, because models need numbers, not text.
dummies = pd.get_dummies(df["channel"], prefix="ch", dtype=int)
df = pd.concat([df, dummies], axis=1)          # axis=1 = add as new columns (side by side)

# .filter(regex=...) selects columns whose names match the pattern, to check the new ones.
df.filter(regex="age_group|bio_minmax|staff_z|ch_").head()

## Step 4 · Descriptive statistics and grouping

In [ ]:
# value_counts(): how many rows per category; to_frame() shows it as a table.
display(df["channel"].value_counts().to_frame())
# groupby + agg: several statistics per group at once.
display(df.groupby("channel")["processing_days"].agg(["mean", "median", "count"]).round(2))
# pivot_table: regions as rows, channels as columns, average days in each cell.
df.pivot_table(index="region", columns="channel", values="processing_days", aggfunc="mean").round(1)

## Step 5 · Correlation and significance
`r` near ±1 = strong relationship; `p` < 0.05 = unlikely to be chance.

In [ ]:
from scipy import stats                          # statistics functions

num = ["age", "staff", "biometric_quality", "docs_complete", "processing_days"]
display(df[num].corr().round(2))                 # correlation of every pair of columns (-1..+1)

for col in ["staff", "biometric_quality", "age"]:
    # pearsonr returns two numbers: r (strength and direction) and p (significance).
    r, p = stats.pearsonr(df[col], df["processing_days"])
    # :+.2f shows a sign and 2 decimals; :.3g is a compact format for tiny p-values.
    print(f"{col:18s} r={r:+.2f}  p={p:.3g}")

## Step 6 · ANOVA: do the channels differ?

In [ ]:
# Make a list with one Series of processing_days per channel ( _ ignores the group name).
groups = [g["processing_days"] for _, g in df.groupby("channel")]
# One-way ANOVA tests whether the group averages differ more than chance would explain.
# The * "unpacks" the list so each group is passed as a separate input.
f, p = stats.f_oneway(*groups)
print(f"F={f:.2f}, p={p:.3f} ->", "different" if p < .05 else "no real difference")

## Step 7 · Simple linear regression: one input

In [ ]:
from sklearn.linear_model import LinearRegression

X1 = df[["staff"]]                               # inputs must be 2-D (a table), hence [[ ]]
y = df["processing_days"]                        # the target is 1-D (a Series)
slr = LinearRegression().fit(X1, y)              # learn the best straight line
# intercept_ = predicted days with 0 staff; coef_[0] = change in days per extra staff member.
print(f"days = {slr.intercept_:.2f} + ({slr.coef_[0]:.3f} x staff)")
print("R²:", round(slr.score(X1, y), 3))         # share of the variation explained (0..1)

## Step 8 · Multiple and polynomial regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

features = ["staff", "biometric_quality", "docs_complete", "age"]
X = df[features]

mlr = LinearRegression().fit(X, y)               # MULTIPLE regression: several inputs at once
print("Multiple R²:", round(mlr.score(X, y), 3), dict(zip(features, mlr.coef_.round(2))))

# A PIPELINE chains steps so they always run in the same order:
poly = Pipeline([
    ("scale", StandardScaler()),                                # 1. put features on the same scale
    ("poly", PolynomialFeatures(degree=2, include_bias=False)), # 2. add squares and pairwise products
    ("model", LinearRegression()),                              # 3. fit a linear model on those
]).fit(X, y)
print("Polynomial R²:", round(poly.score(X, y), 3))

## Step 9 · Visual checks: regression and residual plots

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(12, 4))    # one row, two charts; figsize in inches
sample = df.sample(600, random_state=0)          # plot a random 600 rows so points don't pile up
# regplot: scatter plus the fitted line; alpha=.3 makes points see-through.
sns.regplot(x="biometric_quality", y="processing_days", data=sample,
            scatter_kws={"alpha": .3}, ax=ax[0])
# residplot: prediction errors. A shapeless cloud around 0 means the linear model fits well.
sns.residplot(x=mlr.predict(X), y=y, scatter_kws={"alpha": .2}, ax=ax[1])
ax[1].set(xlabel="Predicted", ylabel="Residual", title="Residuals should look random")
plt.tight_layout()                               # stop labels overlapping
plt.show()

## Step 10 · Evaluate on unseen data: train/test split and cross-validation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 80% to train, 20% held back to test. random_state makes it repeatable.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
mlr.fit(X_tr, y_tr)                              # learn only from the training rows
pred = mlr.predict(X_te)                         # predict the test rows the model never saw

print("Test R²:", round(r2_score(y_te, pred), 3),
      "| RMSE:", round(mean_squared_error(y_te, pred) ** .5, 2),   # ** .5 = square root; error in days
      "| MAE:", round(mean_absolute_error(y_te, pred), 2))         # average absolute error in days

# CROSS-VALIDATION: split into 5 parts, train on 4 and test on 1, five times; average the scores.
print("5-fold CV R²:", cross_val_score(LinearRegression(), X, y, cv=5).mean().round(3))

## Step 11 · Overfitting check, then Ridge + Grid Search

In [ ]:
# Higher polynomial degree = more flexible model. Watch train R² rise while test R² stalls or falls:
# that gap is OVERFITTING (memorising the training data instead of learning the pattern).
for deg in [1, 2, 3, 5]:
    pm = Pipeline([("s", StandardScaler()), ("p", PolynomialFeatures(deg)),
                   ("m", LinearRegression())]).fit(X_tr, y_tr)
    print(f"degree {deg}: train R² {pm.score(X_tr, y_tr):.3f}  test R² {pm.score(X_te, y_te):.3f}")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# RIDGE regression shrinks coefficients to reduce overfitting; alpha = how strongly.
pipe = Pipeline([("s", StandardScaler()), ("p", PolynomialFeatures(2)), ("r", Ridge())])
# GRID SEARCH tries every alpha with 5-fold CV and keeps the best.
# "r__alpha" = the alpha setting of the step named "r" (two underscores).
grid = GridSearchCV(pipe, {"r__alpha": [0.01, 0.1, 1, 10, 100]}, cv=5).fit(X_tr, y_tr)
print("Best:", grid.best_params_, "| test R²:", round(grid.score(X_te, y_te), 3))

## ✍️ Your turn
1. Add `month` and the `ch_` dummy columns to `features`. Does test R² improve?
2. Which region has the highest median processing time? Is the difference significant (ANOVA)?

In [ ]:
# your answer here